# Camada Gold — Data Quality Monitoring

Este notebook lê a tabela Delta `squad1.dq_monitoring_logs` e a Silver `squad1.silver_ecommerce_clientes` para gerar tabelas Gold de monitoramento de qualidade de dados.

Saídas geradas:

- `squad1.gold_dq_resumo_por_regra`: percentual de falha por regra por dia.
- `squad1.gold_dq_resumo_por_tabela`: percentual de registros limpos por tabela por hora.

As tabelas são gravadas como Delta no Databricks e também replicadas para o SQL Server Azure para consumo no Looker.

Estratégia anti-duplicidade: **full refresh idempotente**. As tabelas Gold são recalculadas a partir das fontes oficiais e gravadas com `overwrite`, evitando duplicidade mesmo quando o notebook é executado várias vezes ou por mais de uma pessoa.


In [0]:
# Se este notebook estiver em ingestion/gold.ipynb, execute o config assim:
# %run ../config/config

from pyspark.sql import functions as F
from pyspark.sql.types import *

## Parâmetros das tabelas

Execute o `config` antes deste notebook ou descomente o `%run` abaixo se o caminho estiver correto.


In [0]:
DQ_LOGS_TABLE = "squad1.dq_monitoring_logs"
SILVER_CLIENTES_TABLE = "squad1.silver_ecommerce_clientes"

GOLD_REGRA_TABLE = "squad1.gold_dq_resumo_por_regra"
GOLD_TABELA_TABLE = "squad1.gold_dq_resumo_por_tabela"

SQL_SCHEMA = "squad1"
SQL_GOLD_REGRA = "gold_dq_resumo_por_regra"
SQL_GOLD_TABELA = "gold_dq_resumo_por_tabela"


## Funções utilitárias

In [0]:
def tabela_delta_existe(nome_tabela: str) -> bool:
    return spark.catalog.tableExists(nome_tabela)


def validar_tabela_obrigatoria(nome_tabela: str):
    if not tabela_delta_existe(nome_tabela):
        raise Exception(
            f"Tabela obrigatória não encontrada: {nome_tabela}. "
            "Execute as camadas anteriores antes da Gold."
        )


def gravar_delta_gold(df, nome_tabela: str):
    # Grava a Gold como Delta gerenciada, usando full refresh idempotente.
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )
    print(f"Tabela Delta atualizada: {nome_tabela}")


def gravar_sqlserver_gold(df, schema: str, tabela: str):
    # Replica a Gold para SQL Server usando overwrite para evitar duplicidade.
    tabela_destino = f"{schema}.{tabela}"

    (
        df.write
        .format("sqlserver")
        .mode("overwrite")
        .option("host", JDBC_HOSTNAME)
        .option("port", "1433")
        .option("database", JDBC_DATABASE)
        .option("user", JDBC_USERNAME)
        .option("password", JDBC_PASSWORD)
        .option("dbtable", tabela_destino)
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .save()
    )

    print(f"Tabela SQL Server atualizada: {tabela_destino}")


## Ler fontes da Gold


In [0]:
validar_tabela_obrigatoria(DQ_LOGS_TABLE)
validar_tabela_obrigatoria(SILVER_CLIENTES_TABLE)

df_logs = spark.table(DQ_LOGS_TABLE)
df_silver_clientes = spark.table(SILVER_CLIENTES_TABLE)

print("Registros dq_monitoring_logs:", df_logs.count())
print("Registros silver_ecommerce_clientes:", df_silver_clientes.count())

display(df_logs.limit(10))
display(df_silver_clientes.limit(10))


## Gold 1 — Resumo por regra por dia

Resultado esperado:

- data_referencia
- tabela
- regra
- severidade
- qtd_registros_falhos
- qtd_registros_total
- percentual_falha
- gold_processed_at


In [0]:
df_gold_dq_resumo_por_regra = (
    df_logs
    .withColumn("data_referencia", F.to_date(F.col("timestamp_execucao")))
    .groupBy(
        "data_referencia",
        "tabela",
        "regra",
        "severidade"
    )
    .agg(
        F.sum(F.col("qtd_registros_falhos")).cast("int").alias("qtd_registros_falhos"),
        F.sum(F.col("qtd_registros_total")).cast("int").alias("qtd_registros_total")
    )
    .withColumn(
        "percentual_falha",
        F.when(
            F.col("qtd_registros_total") > 0,
            (F.col("qtd_registros_falhos") / F.col("qtd_registros_total")) * 100
        ).otherwise(F.lit(0.0))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

display(df_gold_dq_resumo_por_regra.orderBy("data_referencia", "tabela", "regra"))


## Gold 2 — Resumo por tabela por hora

Esta tabela usa a Silver para calcular a proporção de linhas limpas por hora.

A linha é considerada limpa quando `silver_linha_valida = True`. Caso essa coluna não exista, o notebook calcula a validade pela negação das flags de regra `r*_falhou`.


In [0]:
# Garante coluna de validade da linha, caso ela ainda não exista na Silver
colunas_silver = df_silver_clientes.columns

if "silver_linha_valida" in colunas_silver:
    df_silver_base = df_silver_clientes.withColumn(
        "silver_linha_valida_calc",
        F.col("silver_linha_valida").cast("boolean")
    )
else:
    flags_regras = [c for c in colunas_silver if c.startswith("r") and c.endswith("_falhou")]

    if len(flags_regras) == 0:
        raise Exception(
            "Não encontrei a coluna silver_linha_valida nem flags de regras terminando com _falhou."
        )

    condicao_falha = None
    for c in flags_regras:
        expr = F.coalesce(F.col(c).cast("boolean"), F.lit(False))
        condicao_falha = expr if condicao_falha is None else (condicao_falha | expr)

    df_silver_base = df_silver_clientes.withColumn(
        "silver_linha_valida_calc",
        ~condicao_falha
    )

# Define timestamp de referência da Silver
if "silver_processed_at" in df_silver_base.columns:
    coluna_tempo_silver = "silver_processed_at"
elif "bronze_ingested_at" in df_silver_base.columns:
    coluna_tempo_silver = "bronze_ingested_at"
else:
    raise Exception("A Silver precisa ter silver_processed_at ou bronze_ingested_at para agregação por hora.")

df_gold_dq_resumo_por_tabela = (
    df_silver_base
    .withColumn("hora_referencia", F.date_trunc("hour", F.col(coluna_tempo_silver)))
    .withColumn("tabela", F.lit("silver_ecommerce_clientes"))
    .groupBy("hora_referencia", "tabela")
    .agg(
        F.count("*").cast("int").alias("qtd_registros_total"),
        F.sum(F.when(F.col("silver_linha_valida_calc") == True, 1).otherwise(0)).cast("int").alias("qtd_registros_limpos"),
        F.sum(F.when(F.col("silver_linha_valida_calc") == False, 1).otherwise(0)).cast("int").alias("qtd_registros_com_falha")
    )
    .withColumn(
        "percentual_registros_limpos",
        F.when(
            F.col("qtd_registros_total") > 0,
            (F.col("qtd_registros_limpos") / F.col("qtd_registros_total")) * 100
        ).otherwise(F.lit(0.0))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

display(df_gold_dq_resumo_por_tabela.orderBy("hora_referencia", "tabela"))


## Gravar Gold em Delta

As tabelas Gold são recalculadas e gravadas em `overwrite` para evitar duplicidade em reexecuções.


In [0]:
gravar_delta_gold(
    df_gold_dq_resumo_por_regra,
    GOLD_REGRA_TABLE
)

gravar_delta_gold(
    df_gold_dq_resumo_por_tabela,
    GOLD_TABELA_TABLE
)


## Replicar Gold para SQL Server Azure

As mesmas tabelas são enviadas para SQL Server para consumo no Looker.

Também usamos `overwrite` para manter a carga idempotente e evitar duplicidade.



In [0]:
gravar_sqlserver_gold(
    df_gold_dq_resumo_por_regra,
    SQL_SCHEMA,
    SQL_GOLD_REGRA
)

gravar_sqlserver_gold(
    df_gold_dq_resumo_por_tabela,
    SQL_SCHEMA,
    SQL_GOLD_TABELA
)


## Validação final

In [0]:
print("=" * 80)
print("GOLD CONCLUÍDA COM SUCESSO")
print("=" * 80)
print("Delta:")
print(f"- {GOLD_REGRA_TABLE}: {spark.table(GOLD_REGRA_TABLE).count()} registros")
print(f"- {GOLD_TABELA_TABLE}: {spark.table(GOLD_TABELA_TABLE).count()} registros")

print("
Amostra gold_dq_resumo_por_regra")
display(spark.table(GOLD_REGRA_TABLE).orderBy(F.desc("gold_processed_at")).limit(20))

print("
Amostra gold_dq_resumo_por_tabela")
display(spark.table(GOLD_TABELA_TABLE).orderBy(F.desc("gold_processed_at")).limit(20))
